# Score agent episodes with PySpark in Microsoft Fabric

This notebook distributes `agent-learning` episode scoring across Fabric Spark executors, inspects per-metric results, and can persist the output to a Lakehouse Delta table. Attach a Lakehouse before enabling the final write.

## Install the SDK

Fabric provides the Spark and PySpark runtime. Install the SDK without its `spark` extra so the environment keeps Fabric's matching PySpark version.

In [ ]:
%pip install agent-learning

## Parameters

Set `save_results` to `True` after attaching a Lakehouse. This cell can also be marked as a Fabric parameter cell for pipeline runs.

In [ ]:
save_results = False
output_table = "agent_learning_episode_scores"

## Create episode rows

The adapter accepts an `episode_json` column. JSON keeps nested episode metadata intact without relying on Spark schema inference for heterogeneous dictionaries.

In [ ]:
import json

from pyspark.sql import SparkSession

from agent_learning import Episode, score_episode_dataframe

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

episodes = [
    Episode(
        id="fabric-episode-001",
        agent_id="support-agent",
        task_id="resolve-ticket",
        user_input="Reset access for the customer account",
        assistant_output="Access was reset and the customer was notified.",
        action_id="reset-access",
        execution_status="completed",
        metadata={"correct_action_id": "reset-access", "task_completed": True},
    ),
    Episode(
        id="fabric-episode-002",
        agent_id="support-agent",
        task_id="resolve-ticket",
        user_input="Explain why the invoice increased",
        assistant_output="The invoice increased after additional seats were added.",
        action_id="explain-invoice",
        execution_status="completed",
        metadata={"correct_action_id": "explain-invoice", "task_completed": True},
    ),
]

episode_rows = [(json.dumps(episode.to_dict()),) for episode in episodes]
episode_df = spark.createDataFrame(episode_rows, ["episode_json"])
display(episode_df)

## Score episodes on Spark executors

The adapter creates metric evaluators once per partition and returns one aggregate reward plus serialized metric details per episode. It does not write to process-local stores from executors.

In [ ]:
scored_df = score_episode_dataframe(episode_df).cache()
display(scored_df.orderBy("episode_id"))

## Inspect individual metrics

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, StringType, StructField, StructType

metric_schema = ArrayType(
    StructType(
        [
            StructField("metric", StringType()),
            StructField("score", DoubleType()),
            StructField("normalized", DoubleType()),
            StructField("status", StringType()),
            StructField("reason", StringType()),
            StructField("evaluator", StringType()),
        ]
    )
)

metric_df = (
    scored_df
    .withColumn("metric_result", F.explode(F.from_json("metrics_json", metric_schema)))
    .select(
        "episode_id",
        "agent_id",
        "reward",
        "metric_result.metric",
        "metric_result.normalized",
        "metric_result.status",
        "metric_result.reason",
        "metric_result.evaluator",
    )
)
display(metric_df.orderBy("episode_id", "metric"))

## Optionally save to the attached Lakehouse

In [ ]:
if save_results:
    scored_df.write.format("delta").mode("overwrite").saveAsTable(output_table)
    print(f"Saved scores to {output_table}")
else:
    print("Set save_results=True to persist scores to the attached Lakehouse.")